# Dual Model Setup: Feature Extraction and Age Estimation

This notebook demonstrates the setup and testing of two deep learning models:

1. **EfficientNet-B0** - Used as a feature extractor for extracting high-level features from face images
2. **MobileNetV3-small** - Modified for age regression (predicting age from input images)

The notebook includes:
- Model loading and configuration
- Classifier head modifications for specific tasks
- Forward pass testing with example image
- GPU acceleration setup (CUDA)

## Environment Check

In [1]:
# !nvidia-smi

In [2]:
import torch
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.get_device_name(0))


2.10.0.dev20251011+cu130
13.0
NVIDIA GeForce RTX 5060 Ti


## Import Required Libraries

In [3]:
# Import all required libraries
import torch
import torch.nn as nn
import torchvision.models as models
from torchvision import transforms
from PIL import Image
import os
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

In [4]:
torch.cuda.is_available()

True

## Device Setup and Configuration

In [5]:
# Device Setup for CUDA GPU Acceleration
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
print("-" * 50)

Using device: cuda:0
GPU Name: NVIDIA GeForce RTX 5060 Ti
--------------------------------------------------


## Image Preprocessing Configuration

In [6]:
# Define the standard input size (for ImageNet-pretrained models)
INPUT_SIZE = 224
MEAN = [0.485, 0.456, 0.406]  # ImageNet normalization values
STD = [0.229, 0.224, 0.225]   # ImageNet normalization values

# Image Preprocessing Transformation
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(INPUT_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD)
])

print("✅ Image preprocessing pipeline configured")
print(f"   - Input size: {INPUT_SIZE}x{INPUT_SIZE}")
print(f"   - Normalization: ImageNet standard (mean={MEAN}, std={STD})")

✅ Image preprocessing pipeline configured
   - Input size: 224x224
   - Normalization: ImageNet standard (mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])


## 1. Feature Extractor Model: EfficientNet-B0

In [7]:
# Model for extracting rich, high-level features from face images

MODEL_NAME_FE = 'EfficientNet-B0'
print(f"Loading Feature Extractor: **{MODEL_NAME_FE}**")

# Load pretrained EfficientNet-B0 weights
fe_weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1
feature_extractor_model = models.efficientnet_b0(weights=fe_weights)

# Remove the classification head to use the model as a pure feature extractor
# EfficientNet structure: features (Sequential), avgpool (AdaptiveAvgPool2d), classifier (Sequential)
feature_extractor_model.classifier = nn.Identity()  # Replace classifier with identity layer

# Get the correct feature dimension from the original classifier
original_classifier = models.efficientnet_b0(weights=fe_weights).classifier
feature_dim = original_classifier[1].in_features  # Linear layer input features

print(f"✅ Classifier head removed to retain feature vector of size: {feature_dim}")

# Move model to device and set to evaluation mode
feature_extractor_model = feature_extractor_model.to(device)
feature_extractor_model.eval()

print(f"✅ Model ready for feature extraction (Output size: {feature_dim})")
print("-" * 50)

Loading Feature Extractor: **EfficientNet-B0**
✅ Classifier head removed to retain feature vector of size: 1280
✅ Model ready for feature extraction (Output size: 1280)
--------------------------------------------------


## 2. Age Estimation Model: MobileNetV3-small

In [8]:
# Enhanced Age Estimation Model: MobileNetV3-small for MORPH-II Dataset

MODEL_NAME_AE = 'MobileNetV3-small'
OUTPUT_CLASSES = 1  # Single output for age (regression)
print(f"Loading Age Estimator: **{MODEL_NAME_AE}**")

# Load pretrained MobileNetV3-small weights
ae_weights = models.MobileNet_V3_Small_Weights.IMAGENET1K_V1
age_estimator_model = models.mobilenet_v3_small(weights=ae_weights)

# Get the correct feature dimension for MobileNetV3-small
original_mobilenet = models.mobilenet_v3_small(weights=ae_weights)
num_ftrs = original_mobilenet.classifier[0].in_features  # First layer (Linear) input features
print(f"✅ Original classifier input features: {num_ftrs}")

# Enhanced classifier head for age regression with regularization
# MORPH-II typically has ages ranging from 16-77, so we'll design accordingly
new_classifier = nn.Sequential(
    nn.Dropout(0.3),  # Dropout for regularization
    nn.Linear(in_features=num_ftrs, out_features=128),  # Intermediate layer
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(in_features=128, out_features=1),  # Final regression output
    nn.ReLU()  # Ensure non-negative age predictions
)
age_estimator_model.classifier = new_classifier

print(f"✅ Enhanced classifier head for MORPH-II age regression:")
print(f"   - Dropout layers (0.3, 0.2) for regularization")
print(f"   - Intermediate layer (128 neurons) for better feature learning")
print(f"   - ReLU output activation for non-negative ages")

# Move model to device and set to training mode (for fine-tuning)
age_estimator_model = age_estimator_model.to(device)
age_estimator_model.train()

print(f"✅ Model ready for MORPH-II training")
print("   Recommended: MAE (L1Loss) or Huber Loss for age regression")
print("-" * 50)

Loading Age Estimator: **MobileNetV3-small**
✅ Original classifier input features: 576
✅ Enhanced classifier head for MORPH-II age regression:
   - Dropout layers (0.3, 0.2) for regularization
   - Intermediate layer (128 neurons) for better feature learning
   - ReLU output activation for non-negative ages
✅ Model ready for MORPH-II training
   Recommended: MAE (L1Loss) or Huber Loss for age regression
--------------------------------------------------


## 3. Forward Pass Testing

## UTKFace Dataset Setup for Age Regression

In [9]:
# Additional imports for dataset handling and training
from torch.utils.data import Dataset, DataLoader, random_split
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import glob
import re
from pathlib import Path

print("✅ Additional imports loaded for UTKFace dataset handling")

✅ Additional imports loaded for UTKFace dataset handling


In [10]:
class UTKFaceDataset(Dataset):
    """
    Simplified UTKFace Dataset for Age Regression
    
    UTKFace filename format: [age]_[gender]_[race]_[date&time].jpg
    Example: 39_1_0_20170116174525125.jpg (Age 39, Male, White, timestamp)
    """
    
    def __init__(self, data_dir, transform=None, age_range=(0, 116)):
        """
        Args:
            data_dir (str): Directory containing UTKFace data (with part1, part2, part3 subdirs)
            transform (callable, optional): Transform to be applied on images
            age_range (tuple): Valid age range for filtering (min_age, max_age)
        """
        self.data_dir = Path(data_dir)
        self.transform = transform
        self.age_range = age_range
        
        # Load and validate dataset
        self.image_paths, self.ages = self._load_dataset()
        
        print(f"✅ UTKFace Dataset loaded:")
        print(f"   - Total samples: {len(self.image_paths)}")
        print(f"   - Age range: {min(self.ages):.0f} - {max(self.ages):.0f} years")
        print(f"   - Mean age: {np.mean(self.ages):.1f} ± {np.std(self.ages):.1f}")
        
    def _load_dataset(self):
        """Load and parse UTKFace dataset from subdirectories"""
        image_paths = []
        ages = []
        
        # Check for subdirectories (part1, part2, part3)
        subdirs = ['part1', 'part2', 'part3']
        
        for subdir in subdirs:
            subdir_path = self.data_dir / subdir
            if subdir_path.exists():
                # Find all jpg files in the subdirectory
                jpg_files = list(subdir_path.glob("*.jpg")) + list(subdir_path.glob("*.JPG"))
                
                for img_path in jpg_files:
                    try:
                        # Extract age from filename (first number before underscore)
                        filename = img_path.name
                        age_str = filename.split('_')[0]
                        age = int(age_str)
                        
                        # Filter by age range
                        if self.age_range[0] <= age <= self.age_range[1]:
                            image_paths.append(str(img_path))
                            ages.append(float(age))  # Convert to float for regression
                            
                    except (ValueError, IndexError):
                        # Skip files that don't follow the expected naming convention
                        continue
        
        if not image_paths:
            raise ValueError(f"No valid UTKFace images found in {self.data_dir}")
            
        return image_paths, ages
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        # Load image
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        
        # Get age label
        age = self.ages[idx]
        
        # Apply transforms
        if self.transform:
            image = self.transform(image)
        
        return image, torch.tensor(age, dtype=torch.float32)

    def get_age_statistics(self):
        """Return age distribution statistics"""
        ages_array = np.array(self.ages)
        return {
            'count': len(ages_array),
            'min': ages_array.min(),
            'max': ages_array.max(),
            'mean': ages_array.mean(),
            'std': ages_array.std(),
            'median': np.median(ages_array)
        }

print("✅ UTKFaceDataset class defined successfully")

✅ UTKFaceDataset class defined successfully


In [11]:
# Data augmentation pipeline for age regression training
# Note: We avoid transforms that might change apparent age (like heavy rotation or distortion)

train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(INPUT_SIZE),           # Random crop instead of center crop
    transforms.RandomHorizontalFlip(p=0.5),     # Horizontal flip is age-neutral
    transforms.ColorJitter(                     # Subtle color variations
        brightness=0.1,
        contrast=0.1,
        saturation=0.1,
        hue=0.05
    ),
    transforms.RandomRotation(5),               # Very small rotation (±5 degrees)
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD)
])

# Validation/test transform (no augmentation)
val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(INPUT_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD)
])

print("✅ Data augmentation pipelines configured:")
print("   - Training: Random crop, flip, color jitter, small rotation")
print("   - Validation: Standard resize and center crop only")

✅ Data augmentation pipelines configured:
   - Training: Random crop, flip, color jitter, small rotation
   - Validation: Standard resize and center crop only


In [12]:
def create_data_loaders(data_dir, batch_size=32, train_split=0.8, random_seed=42):
    """
    Create train and validation data loaders for UTKFace dataset
    
    Args:
        data_dir (str): Path to UTKFace dataset directory (containing part1, part2, part3)
        batch_size (int): Batch size for data loaders
        train_split (float): Proportion of data for training (0-1)
        random_seed (int): Random seed for reproducible splits
        
    Returns:
        tuple: (train_loader, val_loader, dataset_stats)
    """
    print(f"🔄 Creating UTKFace data loaders from: {data_dir}")
    
    # Create full dataset with validation transforms first to get statistics
    full_dataset = UTKFaceDataset(data_dir, transform=val_transform)
    dataset_stats = full_dataset.get_age_statistics()
    
    # Calculate split sizes
    total_size = len(full_dataset)
    train_size = int(train_split * total_size)
    val_size = total_size - train_size
    
    print(f"📊 Dataset split: {train_size} train, {val_size} validation")
    
    # Set random seed for reproducible splits
    torch.manual_seed(random_seed)
    
    # Split the dataset
    train_indices, val_indices = train_test_split(
        range(total_size), 
        train_size=train_size, 
        random_state=random_seed,
        stratify=None  # Could implement age-based stratification if needed
    )
    
    # Create separate datasets with appropriate transforms
    train_dataset = UTKFaceDataset(data_dir, transform=train_transform)
    val_dataset = UTKFaceDataset(data_dir, transform=val_transform)
    
    # Create subset datasets
    from torch.utils.data import Subset
    train_subset = Subset(train_dataset, train_indices)
    val_subset = Subset(val_dataset, val_indices)
    
    # Create data loaders
    train_loader = DataLoader(
        train_subset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=2,
        pin_memory=True if device.type == 'cuda' else False
    )
    
    val_loader = DataLoader(
        val_subset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=True if device.type == 'cuda' else False
    )
    
    print(f"✅ Data loaders created successfully")
    print(f"   - Batch size: {batch_size}")
    print(f"   - Train batches: {len(train_loader)}")
    print(f"   - Validation batches: {len(val_loader)}")
    
    return train_loader, val_loader, dataset_stats

print("✅ Data loader creation function defined")

✅ Data loader creation function defined


## Age Regression Evaluation Metrics

In [13]:
def calculate_age_metrics(predictions, targets):
    """
    Calculate comprehensive age regression metrics
    
    Args:
        predictions (torch.Tensor): Predicted ages
        targets (torch.Tensor): Ground truth ages
        
    Returns:
        dict: Dictionary containing various age regression metrics
    """
    # Convert to numpy for easier calculation
    if isinstance(predictions, torch.Tensor):
        predictions = predictions.cpu().numpy()
    if isinstance(targets, torch.Tensor):
        targets = targets.cpu().numpy()
    
    # Flatten arrays
    predictions = predictions.flatten()
    targets = targets.flatten()
    
    # Calculate absolute errors
    abs_errors = np.abs(predictions - targets)
    
    # Basic metrics
    mae = np.mean(abs_errors)
    mse = np.mean((predictions - targets) ** 2)
    rmse = np.sqrt(mse)
    
    # Age accuracy within tolerance ranges (common in age estimation)
    acc_1 = np.mean(abs_errors <= 1.0) * 100  # Within 1 year
    acc_3 = np.mean(abs_errors <= 3.0) * 100  # Within 3 years
    acc_5 = np.mean(abs_errors <= 5.0) * 100  # Within 5 years
    acc_10 = np.mean(abs_errors <= 10.0) * 100  # Within 10 years
    
    # Statistical measures
    mean_error = np.mean(predictions - targets)  # Bias
    std_error = np.std(abs_errors)
    
    return {
        'MAE': mae,
        'MSE': mse,
        'RMSE': rmse,
        'Accuracy_1yr': acc_1,
        'Accuracy_3yr': acc_3,
        'Accuracy_5yr': acc_5,
        'Accuracy_10yr': acc_10,
        'Mean_Error': mean_error,
        'Std_Error': std_error,
        'Max_Error': np.max(abs_errors),
        'Min_Error': np.min(abs_errors)
    }

def print_metrics(metrics, phase=""):
    """Pretty print age regression metrics"""
    if phase:
        print(f"\n📊 {phase} Metrics:")
    else:
        print(f"\n📊 Age Regression Metrics:")
    
    print(f"   MAE (Mean Absolute Error): {metrics['MAE']:.2f} years")
    print(f"   RMSE (Root Mean Squared Error): {metrics['RMSE']:.2f} years")
    print(f"   Mean Error (Bias): {metrics['Mean_Error']:.2f} years")
    print(f"   Std Error: {metrics['Std_Error']:.2f} years")
    print(f"   Max Error: {metrics['Max_Error']:.2f} years")
    print("")
    print(f"   Age Accuracy:")
    print(f"   - Within 1 year:  {metrics['Accuracy_1yr']:.1f}%")
    print(f"   - Within 3 years: {metrics['Accuracy_3yr']:.1f}%")
    print(f"   - Within 5 years: {metrics['Accuracy_5yr']:.1f}%")
    print(f"   - Within 10 years: {metrics['Accuracy_10yr']:.1f}%")

print("✅ Age regression evaluation metrics defined")

✅ Age regression evaluation metrics defined


## Training and Validation Functions

In [14]:
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR, ReduceLROnPlateau
import time
from tqdm import tqdm

def train_age_model(model, train_loader, val_loader, num_epochs=50, 
                   learning_rate=0.001, weight_decay=1e-4, 
                   patience=10, min_delta=0.1):
    """
    Comprehensive training function for age regression
    
    Args:
        model: The MobileNetV3 model for age estimation
        train_loader: Training data loader
        val_loader: Validation data loader
        num_epochs: Maximum number of training epochs
        learning_rate: Initial learning rate
        weight_decay: L2 regularization strength
        patience: Early stopping patience (epochs)
        min_delta: Minimum improvement for early stopping
    
    Returns:
        dict: Training history and best model state
    """
    print(f"🚀 Starting Age Regression Training")
    print(f"   Model: {MODEL_NAME_AE}")
    print(f"   Epochs: {num_epochs}")
    print(f"   Learning Rate: {learning_rate}")
    print(f"   Weight Decay: {weight_decay}")
    print(f"   Patience: {patience}")
    print("-" * 60)
    
    # Loss function - MAE is commonly used for age regression
    # Huber loss is also good as it's less sensitive to outliers
    criterion = nn.SmoothL1Loss()  # Huber loss (smooth L1)
    # criterion = nn.L1Loss()  # Alternative: Pure MAE
    
    # Optimizer with weight decay
    optimizer = optim.AdamW(model.parameters(), 
                           lr=learning_rate, 
                           weight_decay=weight_decay)
    
    # Learning rate scheduler
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, 
                                patience=5, min_lr=1e-6, verbose=True)
    
    # Training history
    history = {
        'train_loss': [],
        'val_loss': [],
        'train_mae': [],
        'val_mae': [],
        'learning_rates': []
    }
    
    # Early stopping variables
    best_val_loss = float('inf')
    best_model_state = None
    epochs_without_improvement = 0
    
    # Training loop
    for epoch in range(num_epochs):
        start_time = time.time()
        
        # Training phase
        model.train()
        train_loss = 0.0
        train_predictions = []
        train_targets = []
        
        train_pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs} [Train]')
        for batch_idx, (images, ages) in enumerate(train_pbar):
            images, ages = images.to(device), ages.to(device)
            
            # Forward pass
            optimizer.zero_grad()
            outputs = model(images).squeeze()
            loss = criterion(outputs, ages)
            
            # Backward pass
            loss.backward()
            optimizer.step()
            
            # Statistics
            train_loss += loss.item()
            train_predictions.extend(outputs.detach().cpu().numpy())
            train_targets.extend(ages.detach().cpu().numpy())
            
            # Update progress bar
            train_pbar.set_postfix({'Loss': f'{loss.item():.3f}'})
        
        # Calculate training metrics
        train_loss /= len(train_loader)
        train_metrics = calculate_age_metrics(train_predictions, train_targets)
        
        # Validation phase
        model.eval()
        val_loss = 0.0
        val_predictions = []
        val_targets = []
        
        with torch.no_grad():
            val_pbar = tqdm(val_loader, desc=f'Epoch {epoch+1}/{num_epochs} [Val]')
            for images, ages in val_pbar:
                images, ages = images.to(device), ages.to(device)
                outputs = model(images).squeeze()
                loss = criterion(outputs, ages)
                
                val_loss += loss.item()
                val_predictions.extend(outputs.cpu().numpy())
                val_targets.extend(ages.cpu().numpy())
                
                val_pbar.set_postfix({'Loss': f'{loss.item():.3f}'})
        
        # Calculate validation metrics
        val_loss /= len(val_loader)
        val_metrics = calculate_age_metrics(val_predictions, val_targets)
        
        # Update learning rate
        scheduler.step(val_loss)
        current_lr = optimizer.param_groups[0]['lr']
        
        # Save history
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_mae'].append(train_metrics['MAE'])
        history['val_mae'].append(val_metrics['MAE'])
        history['learning_rates'].append(current_lr)
        
        # Print epoch summary
        epoch_time = time.time() - start_time
        print(f"\nEpoch {epoch+1}/{num_epochs} Summary ({epoch_time:.1f}s):")
        print(f"  Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
        print(f"  Train MAE: {train_metrics['MAE']:.2f} | Val MAE: {val_metrics['MAE']:.2f}")
        print(f"  Val Acc (±5yr): {val_metrics['Accuracy_5yr']:.1f}% | LR: {current_lr:.2e}")
        
        # Early stopping check
        if val_loss < best_val_loss - min_delta:
            best_val_loss = val_loss
            best_model_state = model.state_dict().copy()
            epochs_without_improvement = 0
            print(f"  ✅ New best model saved! (Val Loss: {val_loss:.4f})")
        else:
            epochs_without_improvement += 1
            print(f"  ⏳ No improvement for {epochs_without_improvement} epochs")
            
        if epochs_without_improvement >= patience:
            print(f"\n🛑 Early stopping triggered after {epoch+1} epochs")
            break
            
        print("-" * 60)
    
    # Load best model
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        print(f"✅ Best model loaded (Val Loss: {best_val_loss:.4f})")
    
    return {
        'model': model,
        'history': history,
        'best_val_loss': best_val_loss,
        'total_epochs': epoch + 1
    }

print("✅ Training function defined successfully")

✅ Training function defined successfully


## Model Saving and Loading

In [15]:
import json
from datetime import datetime

def save_age_model(model, history, dataset_stats, save_path="utkface_mobilenetv3_age_model.pth"):
    """
    Save trained age regression model with metadata
    
    Args:
        model: Trained PyTorch model
        history: Training history dictionary
        dataset_stats: Dataset statistics
        save_path: Path to save the model
    """
    # Create comprehensive checkpoint
    checkpoint = {
        'model_state_dict': model.state_dict(),
        'model_architecture': MODEL_NAME_AE,
        'input_size': INPUT_SIZE,
        'training_history': history,
        'dataset_stats': dataset_stats,
        'training_date': datetime.now().isoformat(),
        'dataset': 'UTKFace',
        'model_config': {
            'dropout_rates': [0.3, 0.2],
            'intermediate_layer_size': 128,
            'output_activation': 'ReLU',
            'num_parameters': sum(p.numel() for p in model.parameters())
        }
    }
    
    torch.save(checkpoint, save_path)
    print(f"✅ Model saved successfully to: {save_path}")
    print(f"   - Model: {MODEL_NAME_AE}")
    print(f"   - Dataset: UTKFace")
    print(f"   - Total parameters: {checkpoint['model_config']['num_parameters']:,}")
    print(f"   - Best validation MAE: {min(history['val_mae']):.2f} years")
    
    return save_path

def load_age_model(model_path, device=None):
    """
    Load trained age regression model
    
    Args:
        model_path: Path to saved model
        device: Device to load model on
        
    Returns:
        tuple: (model, checkpoint_info)
    """
    if device is None:
        device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    
    # Load checkpoint
    checkpoint = torch.load(model_path, map_location=device)
    
    # Recreate model architecture
    if checkpoint['model_architecture'] == 'MobileNetV3-small':
        # Load base model
        ae_weights = models.MobileNet_V3_Small_Weights.IMAGENET1K_V1
        model = models.mobilenet_v3_small(weights=ae_weights)
        
        # Recreate classifier head based on saved config
        num_ftrs = model.classifier[0].in_features
        new_classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(in_features=num_ftrs, out_features=128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(in_features=128, out_features=1),
            nn.ReLU()
        )
        model.classifier = new_classifier
        
        # Load trained weights
        model.load_state_dict(checkpoint['model_state_dict'])
        model = model.to(device)
        model.eval()
        
        dataset_name = checkpoint.get('dataset', 'Unknown')
        print(f"✅ Model loaded successfully from: {model_path}")
        print(f"   - Architecture: {checkpoint['model_architecture']}")
        print(f"   - Dataset: {dataset_name}")
        print(f"   - Training date: {checkpoint['training_date']}")
        print(f"   - Best validation MAE: {min(checkpoint['training_history']['val_mae']):.2f} years")
        
        return model, checkpoint
    else:
        raise ValueError(f"Unknown model architecture: {checkpoint['model_architecture']}")

def predict_age(model, image_path, transform=None):
    """
    Predict age for a single image
    
    Args:
        model: Trained age regression model
        image_path: Path to image file
        transform: Image preprocessing transform
        
    Returns:
        float: Predicted age
    """
    if transform is None:
        transform = val_transform
    
    # Load and preprocess image
    image = Image.open(image_path).convert('RGB')
    input_tensor = transform(image).unsqueeze(0).to(device)
    
    # Predict
    model.eval()
    with torch.no_grad():
        prediction = model(input_tensor).item()
    
    return max(0, prediction)  # Ensure non-negative age

print("✅ Model saving/loading functions defined successfully")

✅ Model saving/loading functions defined successfully


## Complete Training Example for UTKFace

In [16]:
# EXAMPLE: Complete training pipeline for UTKFace dataset
# The dataset should be in the same folder as this notebook: data/utkface/

def run_complete_training_example(utkface_path="data/utkface"):
    """
    Complete example of training MobileNetV3-small on UTKFace for age regression
    
    Args:
        utkface_path: Path to UTKFace dataset directory containing part1, part2, part3 subdirs
    """
    print("🚀 Starting Complete UTKFace Age Regression Training Pipeline")
    print("=" * 70)
    
    try:
        # 1. Create data loaders
        print("📁 Step 1: Creating data loaders...")
        train_loader, val_loader, dataset_stats = create_data_loaders(
            data_dir=utkface_path,
            batch_size=32,
            train_split=0.8,
            random_seed=42
        )
        
        # 2. Display dataset statistics
        print(f"\n📊 Dataset Statistics:")
        for key, value in dataset_stats.items():
            print(f"   {key}: {value:.2f}")
        
        # 3. Train the model
        print(f"\n🏋️ Step 2: Training MobileNetV3-small...")
        training_results = train_age_model(
            model=age_estimator_model,
            train_loader=train_loader,
            val_loader=val_loader,
            num_epochs=50,
            learning_rate=0.001,
            weight_decay=1e-4,
            patience=10
        )
        
        # 4. Save the trained model
        print(f"\n💾 Step 3: Saving trained model...")
        model_path = save_age_model(
            model=training_results['model'],
            history=training_results['history'],
            dataset_stats=dataset_stats,
            save_path=f"utkface_mobilenetv3_best_{datetime.now().strftime('%Y%m%d_%H%M')}.pth"
        )
        
        # 5. Final evaluation
        print(f"\n🎯 Step 4: Final model evaluation...")
        model = training_results['model']
        model.eval()
        
        # Evaluate on validation set
        val_predictions = []
        val_targets = []
        
        with torch.no_grad():
            for images, ages in val_loader:
                images, ages = images.to(device), ages.to(device)
                outputs = model(images).squeeze()
                val_predictions.extend(outputs.cpu().numpy())
                val_targets.extend(ages.cpu().numpy())
        
        final_metrics = calculate_age_metrics(val_predictions, val_targets)
        print_metrics(final_metrics, "Final Validation")
        
        print("\n" + "=" * 70)
        print("✅ TRAINING COMPLETED SUCCESSFULLY!")
        print(f"   Best Model: {model_path}")
        print(f"   Training Epochs: {training_results['total_epochs']}")
        print(f"   Best Validation MAE: {training_results['best_val_loss']:.2f} years")
        print("=" * 70)
        
        return training_results, model_path
        
    except Exception as e:
        print(f"❌ Error during training: {str(e)}")
        raise

# To run training with the UTKFace dataset in data/utkface/, uncomment the following line:
# training_results, saved_model_path = run_complete_training_example("data/utkface")

print("✅ Complete UTKFace training pipeline function defined")
print("📝 To start training:")
print("   1. Ensure UTKFace dataset is in data/utkface/ directory")
print("   2. Make sure part1, part2, part3 subdirectories exist with images")
print("   3. Uncomment and run the training line above")

✅ Complete UTKFace training pipeline function defined
📝 To start training:
   1. Ensure UTKFace dataset is in data/utkface/ directory
   2. Make sure part1, part2, part3 subdirectories exist with images
   3. Uncomment and run the training line above


## Test UTKFace Dataset Loading

In [17]:
# Test UTKFace dataset loading
utkface_data_path = "data/utkface"

# Check if the dataset directory exists
if os.path.exists(utkface_data_path):
    try:
        print(f"🧪 Testing UTKFace dataset loading from: {utkface_data_path}")
        
        # Try to create a small test dataset
        test_dataset = UTKFaceDataset(utkface_data_path, transform=val_transform)
        
        print(f"✅ Dataset loaded successfully!")
        stats = test_dataset.get_age_statistics()
        
        print(f"\n📊 Dataset Statistics:")
        for key, value in stats.items():
            print(f"   {key}: {value:.2f}")
            
        # Test loading a single sample
        if len(test_dataset) > 0:
            sample_image, sample_age = test_dataset[0]
            print(f"\n🖼️ Sample loaded:")
            print(f"   Image shape: {sample_image.shape}")
            print(f"   Sample age: {sample_age.item():.0f} years")
        
    except Exception as e:
        print(f"❌ Error loading UTKFace dataset: {str(e)}")
        print("Please check that:")
        print("   1. data/utkface/ directory exists")
        print("   2. part1, part2, part3 subdirectories exist")
        print("   3. Image files follow UTKFace naming convention: [age]_[gender]_[race]_[timestamp].jpg")
        
else:
    print(f"⚠️ UTKFace dataset directory not found: {utkface_data_path}")
    print("Please ensure the dataset is located at data/utkface/ with subdirectories part1, part2, part3")

🧪 Testing UTKFace dataset loading from: data/utkface
✅ UTKFace Dataset loaded:
   - Total samples: 48212
   - Age range: 1 - 116 years
   - Mean age: 33.0 ± 20.1
✅ Dataset loaded successfully!

📊 Dataset Statistics:
   count: 48212.00
   min: 1.00
   max: 116.00
   mean: 33.05
   std: 20.14
   median: 29.00

🖼️ Sample loaded:
   Image shape: torch.Size([3, 224, 224])
   Sample age: 100 years


## Summary: UTKFace Age Regression with MobileNetV3-small

🎯 **Setup Complete!** Your MobileNetV3-small model is now ready for age regression training with the UTKFace dataset.

### What's Ready:

✅ **MobileNetV3-small Model**: Enhanced with regression head (dropout + intermediate layer + ReLU output)  
✅ **UTKFace Dataset**: Simplified loader for all 48,212 images across part1, part2, part3  
✅ **Data Augmentation**: Age-appropriate transforms (no heavy rotation/distortion)  
✅ **Training Pipeline**: Complete with MAE/Huber loss, early stopping, and progress tracking  
✅ **Evaluation Metrics**: MAE, RMSE, accuracy within 1/3/5/10 years tolerance  
✅ **Model Persistence**: Save/load trained models with full metadata  

### Dataset Info:
- **Total Images**: 48,212 samples
- **Age Range**: 1-116 years (mean: 33.0 ± 20.1)
- **Format**: `[age]_[gender]_[race]_[timestamp].jpg`

### To Start Training:
```python
# Uncomment this line to start training:
# training_results, saved_model_path = run_complete_training_example("data/utkface")
```

The model will automatically:
- Split data 80/20 for train/validation
- Use Huber loss for robust age regression
- Apply early stopping (patience=10)
- Save the best model checkpoint
- Provide comprehensive metrics

In [18]:
# Test both models with an example image

image_path = "test.jpg"  # NOTE: Ensure this image file exists for the test
if not os.path.exists(image_path):
    print(f"⚠️ ERROR: Image file '{image_path}' not found. Skipping forward pass check.")
    print("Please ensure you have a test image file in the current directory.")
else:
    print(f"🖼️ Running Forward Pass on Image: {image_path}")
    
    # Load and preprocess the image
    image = Image.open(image_path).convert("RGB")
    input_batch = preprocess(image).unsqueeze(0).to(device)  # Shape: [1, 3, 224, 224]
    print(f"   Input tensor shape: {input_batch.shape}")

    with torch.no_grad():
        # 1. Feature Extraction Test
        print(f"\n🔍 Testing {MODEL_NAME_FE} Feature Extraction:")
        face_features = feature_extractor_model(input_batch)
        print(f"   ✅ Feature Vector Shape: {face_features.shape}")
        print(f"   ✅ Feature vector extracted successfully")

        # 2. Age Estimation Test
        print(f"\n🎯 Testing {MODEL_NAME_AE} Age Estimation:")
        age_estimator_model.eval()  # Switch to eval mode for prediction
        predicted_age = age_estimator_model(input_batch)
        age_value = predicted_age.item()
        
        print(f"   ✅ Prediction Shape: {predicted_age.shape}")
        print(f"   ✅ Predicted (raw) age value: **{age_value:.2f}**")
        print(f"   ⚠️  Note: This is an untrained model, so the prediction is random")

    print("\n" + "=" * 60)
    print("✅ DUAL-MODEL SETUP COMPLETE")
    print("Both models are ready for the research pipeline!")
    print("=" * 60)

🖼️ Running Forward Pass on Image: test.jpg
   Input tensor shape: torch.Size([1, 3, 224, 224])

🔍 Testing EfficientNet-B0 Feature Extraction:
   ✅ Feature Vector Shape: torch.Size([1, 1280])
   ✅ Feature vector extracted successfully

🎯 Testing MobileNetV3-small Age Estimation:
   ✅ Prediction Shape: torch.Size([1, 1])
   ✅ Predicted (raw) age value: **0.06**
   ⚠️  Note: This is an untrained model, so the prediction is random

✅ DUAL-MODEL SETUP COMPLETE
Both models are ready for the research pipeline!
